# Вводный сценарий: от загрузки данных до сохранения модели

## Сквозной пример на данных о продажах

В этом notebook мы пройдём базовый рабочий цикл аналитика данных и начинающего специалиста по машинному обучению:

1. импортируем библиотеки;
2. загрузим данные из CSV, Excel и JSON;
3. выполним предварительный анализ;
4. подготовим и объединим таблицы;
5. построим простую модель для оценки срока доставки заказа;
6. оценим качество модели;
7. визуализируем результаты;
8. сохраним модель на диск;
9. загрузим сохранённую модель и применим её к новому заказу.

> **Важно.** Это учебный пример полного процесса. Главная цель — понять последовательность действий, а не получить промышленную модель максимального качества.

## Практическая задача

Компания хранит данные в трёх файлах:

- `orders.csv` — заказы;
- `products.xlsx` — справочник товаров;
- `customers.json` — справочник клиентов.

Нужно объединить эти данные и построить модель, которая прогнозирует количество дней доставки нового заказа.

### Почему это задача регрессии

Целевая переменная `delivery_days` — число. Поэтому мы будем решать задачу **регрессии**: модель должна предсказать числовое значение.

## Структура проекта

Notebook использует относительные пути и поэтому подходит для Google Colab и VS Code.

```text
project/
├── data/
│   └── raw/
│       ├── orders.csv
│       ├── products.xlsx
│       └── customers.json
├── models/
├── outputs/
└── 00_intro_sales_data_pipeline_ml.ipynb
```

Если исходных файлов нет, notebook создаст демонстрационные данные автоматически.

# 1. Импорт библиотек

Библиотеки подключаются один раз в начале работы.

- `pandas` — таблицы и обработка данных;
- `numpy` — числовые операции;
- `matplotlib` — графики;
- `scikit-learn` — подготовка признаков, обучение и оценка модели;
- `joblib` — сохранение и загрузка модели;
- `pathlib` — удобная работа с путями.

In [ ]:
from pathlib import Path
import json
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

print("Версия Python:", sys.version.split()[0])
print("Версия pandas:", pd.__version__)

## 1.1. Создаём рабочие папки

`Path.cwd()` возвращает текущую рабочую папку. Все остальные пути строятся относительно неё.

In [ ]:
PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

ORDERS_PATH = RAW_DIR / "orders.csv"
PRODUCTS_PATH = RAW_DIR / "products.xlsx"
CUSTOMERS_PATH = RAW_DIR / "customers.json"
MODEL_PATH = MODEL_DIR / "delivery_days_model.joblib"

print("Рабочая папка:", PROJECT_ROOT)
print("Исходные данные:", RAW_DIR)
print("Результаты:", OUTPUT_DIR)
print("Модели:", MODEL_DIR)

# 2. Резервное создание демонстрационных данных

В комплекте уже есть исходные файлы. Ячейка ниже нужна для Google Colab, когда был загружен только notebook.

Она создаёт три файла разных форматов. Данные синтетические и не содержат персональной информации.

In [ ]:
def create_demo_sales_data(raw_dir: Path, seed: int = 42) -> None:
    """Создаёт учебные файлы orders.csv, products.xlsx и customers.json."""
    rng = np.random.default_rng(seed)

    products = pd.DataFrame([
        ("P001", "Ноутбук Lite 14", "Электроника", 64990.0, 46500.0),
        ("P002", "Ноутбук Pro 15", "Электроника", 94990.0, 70500.0),
        ("P003", "Смартфон Start", "Электроника", 29990.0, 21400.0),
        ("P004", "Смартфон Plus", "Электроника", 54990.0, 40200.0),
        ("P005", "Кресло Office", "Офис", 18990.0, 11900.0),
        ("P006", "Стол Compact", "Офис", 15990.0, 9800.0),
        ("P007", "Лампа Desk", "Офис", 4990.0, 2650.0),
        ("P008", "Кофемашина Home", "Бытовая техника", 38990.0, 27400.0),
        ("P009", "Пылесос Smart", "Бытовая техника", 26990.0, 18300.0),
        ("P010", "Очиститель воздуха", "Бытовая техника", 21990.0, 14700.0),
        ("P011", "Наушники Air", "Аксессуары", 8990.0, 4650.0),
        ("P012", "Мышь Wireless", "Аксессуары", 3490.0, 1750.0),
    ], columns=["product_id", "product_name", "category", "unit_price", "cost_price"])
    products.to_excel(raw_dir / "products.xlsx", index=False, sheet_name="products")

    cities_by_region = {
        "Центр": ["Москва", "Тула", "Ярославль"],
        "Северо-Запад": ["Санкт-Петербург", "Псков", "Архангельск"],
        "Юг": ["Ростов-на-Дону", "Краснодар", "Сочи"],
        "Урал": ["Екатеринбург", "Челябинск", "Пермь"],
        "Сибирь": ["Новосибирск", "Омск", "Красноярск"],
        "Дальний Восток": ["Владивосток", "Хабаровск", "Якутск"],
    }
    region_names = list(cities_by_region)

    customers = []
    for i in range(1, 181):
        region = rng.choice(region_names, p=[0.28, 0.16, 0.15, 0.16, 0.15, 0.10])
        registration_date = pd.Timestamp("2022-01-01") + pd.to_timedelta(
            int(rng.integers(0, 1095)), unit="D"
        )
        customers.append({
            "customer_id": f"C{i:04d}",
            "segment": rng.choice(["Массовый", "Премиум", "Корпоративный"], p=[0.62, 0.23, 0.15]),
            "region": region,
            "city": rng.choice(cities_by_region[region]),
            "registration_date": registration_date.strftime("%Y-%m-%d"),
            "loyalty_level": rng.choice(["Bronze", "Silver", "Gold"], p=[0.48, 0.35, 0.17]),
        })

    for index in [7, 51, 119]:
        customers[index]["segment"] = None

    with open(raw_dir / "customers.json", "w", encoding="utf-8") as file:
        json.dump(customers, file, ensure_ascii=False, indent=2)

    region_effect = {
        "Центр": 0.0,
        "Северо-Запад": 0.8,
        "Юг": 1.1,
        "Урал": 1.8,
        "Сибирь": 2.7,
        "Дальний Восток": 4.0,
    }
    category_effect = {
        "Электроника": 0.7,
        "Офис": 0.5,
        "Бытовая техника": 1.2,
        "Аксессуары": 0.0,
    }
    channel_effect = {"Онлайн": 0.7, "Магазин": 0.0, "Партнёр": 1.0}
    segment_effect = {"Массовый": 0.4, "Премиум": -0.2, "Корпоративный": -0.4, None: 0.2}

    product_ids = products["product_id"].tolist()
    customer_ids = [row["customer_id"] for row in customers]
    order_rows = []

    for i in range(1, 901):
        order_date = pd.Timestamp("2025-01-01") + pd.to_timedelta(
            int(rng.integers(0, 365)), unit="D"
        )
        customer_id = rng.choice(customer_ids)
        product_id = rng.choice(product_ids)
        quantity = int(rng.integers(1, 7))
        discount = float(rng.choice([0.00, 0.05, 0.10, 0.15, 0.20], p=[0.28, 0.24, 0.25, 0.16, 0.07]))
        channel = rng.choice(["Онлайн", "Магазин", "Партнёр"], p=[0.56, 0.29, 0.15])

        customer = customers[int(customer_id[1:]) - 1]
        product = products.loc[products["product_id"].eq(product_id)].iloc[0]
        noise = float(rng.normal(0, 0.9))
        delivery_days = round(
            2.3
            + region_effect[customer["region"]]
            + category_effect[product["category"]]
            + channel_effect[channel]
            + segment_effect[customer["segment"]]
            + max(quantity - 3, 0) * 0.35
            + noise
        )

        order_rows.append({
            "order_id": f"ORD-{10000 + i}",
            "order_date": order_date.strftime("%Y-%m-%d"),
            "customer_id": customer_id,
            "product_id": product_id,
            "quantity": quantity,
            "discount": discount,
            "channel": channel,
            "delivery_days": int(np.clip(delivery_days, 1, 14)),
        })

    orders = pd.DataFrame(order_rows)
    orders.loc[[34, 212, 488, 703], "discount"] = np.nan
    orders = pd.concat([orders, orders.iloc[[10, 20]]], ignore_index=True)
    orders.to_csv(raw_dir / "orders.csv", index=False, encoding="utf-8-sig")


required_files = [ORDERS_PATH, PRODUCTS_PATH, CUSTOMERS_PATH]
if not all(path.exists() for path in required_files):
    print("Не все файлы найдены. Создаём демонстрационные данные...")
    create_demo_sales_data(RAW_DIR)
else:
    print("Исходные файлы уже существуют. Повторное создание не требуется.")

## 2.1. Проверяем наличие файлов

Этот блок проверяет готовность проекта. Он не выполняет аналитические расчёты.

In [ ]:
for file_path in [ORDERS_PATH, PRODUCTS_PATH, CUSTOMERS_PATH]:
    status = "OK" if file_path.exists() else "НЕ НАЙДЕН"
    print(f"{status}: {file_path}")

# 3. Загрузка данных разных форматов

Каждый формат загружается своей функцией:

- CSV — `pd.read_csv()`;
- Excel — `pd.read_excel()`;
- JSON — `pd.read_json()`.

In [ ]:
orders = pd.read_csv(
    ORDERS_PATH,
    dtype={
        "order_id": "string",
        "customer_id": "string",
        "product_id": "string",
    },
)

products = pd.read_excel(
    PRODUCTS_PATH,
    sheet_name="products",
    dtype={"product_id": "string"},
)

customers = pd.read_json(
    CUSTOMERS_PATH,
    dtype={"customer_id": "string"},
)

print("Заказы:", orders.shape)
print("Товары:", products.shape)
print("Клиенты:", customers.shape)

# 4. Предварительный анализ данных

До объединения таблиц нужно понять:

- какие столбцы доступны;
- сколько строк и столбцов;
- какие типы данных распознаны;
- есть ли пропуски;
- есть ли дубликаты;
- уникальны ли ключи в справочниках.

In [ ]:
print("Первые строки orders:")
display(orders.head())

print("Первые строки products:")
display(products.head())

print("Первые строки customers:")
display(customers.head())

In [ ]:
def data_quality_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Возвращает компактный отчёт о типах и пропусках."""
    report = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "missing": df.isna().sum().values,
        "unique": df.nunique(dropna=True).values,
    })
    print(f"Таблица {name}: {df.shape[0]} строк, {df.shape[1]} столбцов")
    return report

print("Качество orders:")
display(data_quality_report(orders, "orders"))

print("Качество products:")
display(data_quality_report(products, "products"))

print("Качество customers:")
display(data_quality_report(customers, "customers"))

In [ ]:
print("Дубликаты order_id:", orders["order_id"].duplicated().sum())
print("Дубликаты product_id:", products["product_id"].duplicated().sum())
print("Дубликаты customer_id:", customers["customer_id"].duplicated().sum())

# 5. Подготовка данных

На этом этапе мы:

1. удалим повторяющиеся заказы;
2. преобразуем даты;
3. заполним пропуски скидки;
4. очистим текстовые поля;
5. проверим уникальность справочников.

In [ ]:
# 1. Удаляем дубликаты основной сущности — заказа.
orders_clean = orders.drop_duplicates(subset="order_id", keep="first").copy()

# 2. Преобразуем даты из текста в datetime.
orders_clean["order_date"] = pd.to_datetime(orders_clean["order_date"], errors="coerce")
customers["registration_date"] = pd.to_datetime(customers["registration_date"], errors="coerce")

# 3. Пропуск скидки интерпретируем как отсутствие скидки.
orders_clean["discount"] = orders_clean["discount"].fillna(0.0)

# 4. Удаляем случайные пробелы в текстовых полях.
for column in ["customer_id", "product_id", "channel"]:
    orders_clean[column] = orders_clean[column].astype("string").str.strip()

products["product_id"] = products["product_id"].astype("string").str.strip()
customers["customer_id"] = customers["customer_id"].astype("string").str.strip()
customers["segment"] = customers["segment"].fillna("Неизвестный")

print("Строк до удаления дубликатов:", len(orders))
print("Строк после удаления дубликатов:", len(orders_clean))
print("Пропуски скидки после обработки:", orders_clean["discount"].isna().sum())

## 5.1. Проверяем бизнес-правила

Для количества, цены и срока доставки допустимы только положительные значения. Скидка должна находиться в диапазоне от 0 до 1.

In [ ]:
assert orders_clean["quantity"].gt(0).all(), "Найдено неположительное количество"
assert orders_clean["discount"].between(0, 1).all(), "Скидка вне диапазона 0–1"
assert orders_clean["delivery_days"].gt(0).all(), "Срок доставки должен быть положительным"
assert products["unit_price"].gt(0).all(), "Цена товара должна быть положительной"

print("Базовые бизнес-правила соблюдены.")

# 6. Интеграция данных

`orders` — таблица фактов: одна строка соответствует одному заказу.

`products` и `customers` — справочники. Один товар и один клиент могут встречаться во многих заказах.

Используем `left merge`, чтобы сохранить все заказы. Параметр `validate="many_to_one"` проверяет ожидаемую связь: много заказов к одной строке справочника.

In [ ]:
rows_before = len(orders_clean)

sales = orders_clean.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one",
)

sales = sales.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one",
)

print("Строк до объединения:", rows_before)
print("Строк после объединения:", len(sales))
print("Заказов без найденного товара:", sales["product_name"].isna().sum())
print("Заказов без найденного клиента:", sales["region"].isna().sum())

## 6.1. Создаём расчётные поля

После интеграции доступны цена, себестоимость и характеристики клиента. Теперь можно рассчитать показатели продаж.

In [ ]:
sales["gross_revenue"] = sales["quantity"] * sales["unit_price"]
sales["revenue"] = sales["gross_revenue"] * (1 - sales["discount"])
sales["cost"] = sales["quantity"] * sales["cost_price"]
sales["profit"] = sales["revenue"] - sales["cost"]
sales["order_month"] = sales["order_date"].dt.month
sales["order_month_name"] = sales["order_date"].dt.to_period("M").astype(str)

selected_columns = [
    "order_id", "order_date", "customer_id", "product_id",
    "category", "segment", "region", "channel", "quantity",
    "discount", "revenue", "profit", "delivery_days",
]
display(sales[selected_columns].head())

# 7. Небольшой аналитический обзор

Перед моделированием полезно посмотреть на общие показатели и распределения.

In [ ]:
summary = pd.Series({
    "Количество заказов": sales["order_id"].nunique(),
    "Выручка": sales["revenue"].sum(),
    "Прибыль": sales["profit"].sum(),
    "Средний чек": sales["revenue"].mean(),
    "Средний срок доставки": sales["delivery_days"].mean(),
})

display(summary.to_frame("value"))

In [ ]:
monthly_sales = (
    sales.groupby("order_month_name", as_index=False)
    .agg(revenue=("revenue", "sum"), orders=("order_id", "nunique"))
    .sort_values("order_month_name")
)

plt.figure(figsize=(10, 4))
plt.plot(monthly_sales["order_month_name"], monthly_sales["revenue"], marker="o")
plt.title("Динамика выручки по месяцам")
plt.xlabel("Месяц")
plt.ylabel("Выручка")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
category_sales = (
    sales.groupby("category", as_index=False)
    .agg(revenue=("revenue", "sum"))
    .sort_values("revenue", ascending=False)
)

plt.figure(figsize=(8, 4))
plt.bar(category_sales["category"], category_sales["revenue"])
plt.title("Выручка по категориям")
plt.xlabel("Категория")
plt.ylabel("Выручка")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# 8. Подготовка данных для модели

Будем прогнозировать `delivery_days`.

### Числовые признаки

- количество;
- скидка;
- выручка;
- месяц заказа.

### Категориальные признаки

- канал;
- категория товара;
- сегмент клиента;
- регион;
- уровень лояльности.

Идентификаторы не используем: они обозначают объекты, но обычно не несут устойчивого смысла для нового заказа.

In [ ]:
target = "delivery_days"

numeric_features = [
    "quantity",
    "discount",
    "revenue",
    "order_month",
]

categorical_features = [
    "channel",
    "category",
    "segment",
    "region",
    "loyalty_level",
]

feature_columns = numeric_features + categorical_features

X = sales[feature_columns].copy()
y = sales[target].copy()

print("Матрица признаков:", X.shape)
print("Целевая переменная:", y.shape)
display(X.head())

## 8.1. Делим данные на обучающую и тестовую части

- обучающая выборка нужна для построения модели;
- тестовая выборка нужна для честной проверки на данных, которые модель не использовала при обучении.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

print("Обучающая выборка:", X_train.shape)
print("Тестовая выборка:", X_test.shape)

# 9. Создание конвейера подготовки и моделирования

Категориальные значения нельзя напрямую передать большинству моделей. `OneHotEncoder` превращает категории в числовые индикаторы.

`Pipeline` объединяет подготовку данных и модель в один объект. Это особенно важно при сохранении модели: вместе сохраняются и правила преобразования признаков.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features),
    ]
)

model = RandomForestRegressor(
    n_estimators=180,
    max_depth=10,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1,
)

model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model),
])

model_pipeline

# 10. Обучение модели

Метод `fit()` находит закономерности в обучающей выборке.

In [ ]:
model_pipeline.fit(X_train, y_train)
print("Модель обучена.")

# 11. Оценка качества модели

Используем три метрики:

- **MAE** — средняя абсолютная ошибка в днях;
- **RMSE** — сильнее штрафует крупные ошибки;
- **R²** — доля объяснённой вариативности. Чем ближе к 1, тем лучше модель описывает данные.

In [ ]:
y_pred = model_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

metrics = pd.DataFrame({
    "metric": ["MAE", "RMSE", "R2"],
    "value": [mae, rmse, r2],
})

display(metrics)
print(f"В среднем модель ошибается примерно на {mae:.2f} дня.")

## 11.1. Сравниваем фактические и прогнозные значения

Точки ближе к диагонали означают более точные прогнозы.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.6)

min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())
plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")

plt.title("Фактический и прогнозный срок доставки")
plt.xlabel("Фактические дни")
plt.ylabel("Прогноз модели")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 11.2. Анализируем ошибки

Остаток — разница между фактическим и прогнозным значением. Распределение вокруг нуля означает отсутствие сильного систематического смещения.

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(8, 4))
plt.hist(residuals, bins=20)
plt.title("Распределение ошибок модели")
plt.xlabel("Фактическое значение минус прогноз")
plt.ylabel("Количество наблюдений")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 12. Сохраняем результаты прогнозирования

Файл с прогнозами полезен для проверки, отчёта или дальнейшей визуализации.

In [ ]:
predictions = X_test.copy()
predictions["actual_delivery_days"] = y_test.values
predictions["predicted_delivery_days"] = y_pred
predictions["absolute_error"] = np.abs(
    predictions["actual_delivery_days"] - predictions["predicted_delivery_days"]
)

predictions_path = OUTPUT_DIR / "delivery_predictions.csv"
predictions.to_csv(predictions_path, index=False, encoding="utf-8-sig")

print("Файл сохранён:", predictions_path)
display(predictions.head())

# 13. Сохранение модели

Сохраняем весь `Pipeline`, а не только алгоритм. Поэтому вместе с моделью сохраняются:

- список используемых признаков;
- заполнение пропусков;
- кодирование категорий;
- обученный алгоритм.

In [ ]:
joblib.dump(model_pipeline, MODEL_PATH)

print("Модель сохранена:", MODEL_PATH)
print("Размер файла, КБ:", round(MODEL_PATH.stat().st_size / 1024, 1))

# 14. Загрузка модели и применение к новому заказу

На практике обученная модель обычно загружается отдельным приложением или сервисом. Здесь мы имитируем этот процесс в notebook.

In [ ]:
loaded_model = joblib.load(MODEL_PATH)
print("Модель успешно загружена.")

In [ ]:
new_order = pd.DataFrame([{
    "quantity": 3,
    "discount": 0.10,
    "revenue": 53973.0,
    "order_month": 7,
    "channel": "Онлайн",
    "category": "Офис",
    "segment": "Премиум",
    "region": "Урал",
    "loyalty_level": "Silver",
}])

predicted_days = loaded_model.predict(new_order)[0]

print("Новый заказ:")
display(new_order)
print(f"Прогнозный срок доставки: {predicted_days:.1f} дня")

## 14.1. Простая функция для применения модели

Функция скрывает технические детали загрузки и возвращает понятный результат.

In [ ]:
def predict_delivery_days(model_path: Path, order_data: pd.DataFrame) -> pd.DataFrame:
    """Загружает модель и добавляет прогноз срока доставки."""
    model = joblib.load(model_path)
    result = order_data.copy()
    result["predicted_delivery_days"] = model.predict(order_data)
    return result

new_order_result = predict_delivery_days(MODEL_PATH, new_order)
display(new_order_result)

# 15. Сохранение интегрированного датасета

Подготовленный датасет можно использовать в следующем занятии, BI-системе или дополнительном анализе.

In [ ]:
integrated_data_path = OUTPUT_DIR / "sales_integrated.csv"
sales.to_csv(integrated_data_path, index=False, encoding="utf-8-sig")

print("Интегрированный датасет сохранён:", integrated_data_path)
print("Размер:", sales.shape)

# 16. Итоги

В этом notebook мы прошли полный базовый цикл:

- импортировали библиотеки;
- загрузили CSV, Excel и JSON;
- проверили структуру и качество данных;
- удалили дубликаты и обработали пропуски;
- объединили таблицы по ключам;
- рассчитали показатели продаж;
- подготовили числовые и категориальные признаки;
- обучили модель регрессии;
- оценили MAE, RMSE и R²;
- построили графики;
- сохранили прогнозы;
- сохранили и повторно загрузили модель;
- применили модель к новому заказу.

## Ограничения учебной модели

- данные синтетические;
- признаки выбраны для демонстрации;
- не выполнялся подбор гиперпараметров;
- не проверялась устойчивость во времени;
- перед реальным внедрением потребуются мониторинг качества, контроль входных данных и регулярное переобучение.

## Чек-лист самопроверки

- [ ] Все ячейки выполнены сверху вниз без ошибок.
- [ ] Три исходных файла найдены или созданы.
- [ ] После объединения количество заказов не изменилось.
- [ ] Созданы поля `revenue`, `profit` и `order_month`.
- [ ] Модель обучена на обучающей выборке.
- [ ] Рассчитаны MAE, RMSE и R².
- [ ] Построены графики продаж и качества модели.
- [ ] В папке `outputs` появился файл прогнозов.
- [ ] В папке `models` появился файл модели.
- [ ] Загруженная модель сформировала прогноз для нового заказа.